<a href="https://colab.research.google.com/github/harshshukla07/Neural-Style-Transfer/blob/main/Neural_style_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install tensorflow numpy matplotlib pillow ipywidgets

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
import PIL.Image
import cv2
from IPython.display import clear_output
from google.colab import files
from ipywidgets import interact, interactive, fixed, IntSlider, FloatSlider, Checkbox, Button, VBox, HBox, Output, Layout
import os
import tempfile

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("Setup complete!")

In [ ]:

def tensor_to_image(tensor):
    """Convert a tensor to a PIL image"""
    tensor = tensor * 255
    tensor = np.array(tensor, dtype=np.uint8)
    if np.ndim(tensor) > 3:
        tensor = tensor[0]
    return PIL.Image.fromarray(tensor)

def load_img(path_to_img, max_dim=512):
    """Load and preprocess an image with better error handling"""
    try:
        # First try using PIL to validate the image
        pil_img = PIL.Image.open(path_to_img).convert('RGB')

        # Convert PIL image to TensorFlow tensor
        img_array = np.array(pil_img, dtype=np.float32) / 255.0
        img = tf.constant(img_array)

        # Calculate new dimensions while preserving aspect ratio
        shape = tf.cast(tf.shape(img)[:-1], tf.float32)
        long_dim = max(shape)
        scale = max_dim / long_dim
        new_shape = tf.cast(shape * scale, tf.int32)

        # Resize the image
        img = tf.image.resize(img, new_shape)
        img = img[tf.newaxis, :]

        return img

    except Exception as e:
        print(f"Error loading image {path_to_img}: {e}")
        print("Attempting to fix with TensorFlow's decoder...")

        try:
            # Fall back to TensorFlow's decoder
            img = tf.io.read_file(path_to_img)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.convert_image_dtype(img, tf.float32)

            # Calculate new dimensions while preserving aspect ratio
            shape = tf.cast(tf.shape(img)[:-1], tf.float32)
            long_dim = max(shape)
            scale = max_dim / long_dim
            new_shape = tf.cast(shape * scale, tf.int32)

            # Resize the image
            img = tf.image.resize(img, new_shape)
            img = img[tf.newaxis, :]

            return img
        except Exception as e2:
            print(f"Failed to fix image: {e2}")
            raise ValueError(f"Could not load image {path_to_img} - please try a different image file")

def imshow(image, title=None):
    """Display an image with an optional title"""
    if len(image.shape) > 3:
        image = tf.squeeze(image, axis=0)

    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')

def clip_0_1(image):
    """Clip image values between 0 and 1"""
    return tf.clip_by_value(image, clip_value_min=0.0, clip_value_max=1.0)

# Function to display images side by side
def show_images(content_image, style_image, stylized_image=None, figsize=(15, 5)):
    plt.figure(figsize=figsize)

    # Display content image
    plt.subplot(1, 3 if stylized_image is not None else 2, 1)
    imshow(content_image, 'Content Image')

    # Display style image
    plt.subplot(1, 3 if stylized_image is not None else 2, 2)
    imshow(style_image, 'Style Image')

    # Display stylized image if provided
    if stylized_image is not None:
        plt.subplot(1, 3, 3)
        imshow(stylized_image, 'Stylized Image')

    plt.tight_layout()
    plt.show()

# Image upload function
def upload_and_process_images(max_dim=512):
    """Upload content and style images and prepare them for style transfer with enhanced error handling"""
    # Create temporary directories
    content_dir = tempfile.mkdtemp()
    style_dir = tempfile.mkdtemp()

    # Upload content image
    print("Upload a content image (JPEG, PNG, GIF, or BMP format):")
    uploaded_content = files.upload()
    content_path = None

    for filename, data in uploaded_content.items():
        content_path = os.path.join(content_dir, filename)
        with open(content_path, 'wb') as f:
            f.write(data)
        print(f"Uploaded content image: {filename}")

        # Try validating the image
        try:
            PIL.Image.open(content_path).verify()
        except Exception as e:
            print(f"Warning: The content image may have format issues: {e}")
            print("Attempting to fix by converting to JPEG...")
            try:
                # Try opening and resaving as JPEG
                img = PIL.Image.open(content_path).convert('RGB')
                fixed_path = os.path.join(content_dir, "fixed_content.jpg")
                img.save(fixed_path)
                content_path = fixed_path
                print("Fixed content image saved as JPEG")
            except Exception as e2:
                print(f"Could not fix content image: {e2}")

    if content_path is None:
        print("No content image uploaded!")
        return None, None

    # Upload style image
    print("\nUpload a style image (JPEG, PNG, GIF, or BMP format):")
    uploaded_style = files.upload()
    style_path = None

    for filename, data in uploaded_style.items():
        style_path = os.path.join(style_dir, filename)
        with open(style_path, 'wb') as f:
            f.write(data)
        print(f"Uploaded style image: {filename}")

        # Try validating the image
        try:
            PIL.Image.open(style_path).verify()
        except Exception as e:
            print(f"Warning: The style image may have format issues: {e}")
            print("Attempting to fix by converting to JPEG...")
            try:
                # Try opening and resaving as JPEG
                img = PIL.Image.open(style_path).convert('RGB')
                fixed_path = os.path.join(style_dir, "fixed_style.jpg")
                img.save(fixed_path)
                style_path = fixed_path
                print("Fixed style image saved as JPEG")
            except Exception as e2:
                print(f"Could not fix style image: {e2}")

    if style_path is None:
        print("No style image uploaded!")
        return None, None

    # Load and preprocess images with enhanced error handling
    try:
        content_image = load_img(content_path, max_dim=max_dim)
        print("Content image loaded successfully")
    except Exception as e:
        print(f"Error loading content image: {e}")
        print("Please try uploading a different content image")
        return None, None

    try:
        style_image = load_img(style_path, max_dim=max_dim)
        print("Style image loaded successfully")
    except Exception as e:
        print(f"Error loading style image: {e}")
        print("Please try uploading a different style image")
        return content_image, None

    # Display the uploaded images
    print("Both images loaded successfully. Displaying preview:")
    show_images(content_image, style_image)

    return content_image, style_image

print("Enhanced image processing functions defined!")

In [ ]:
def vgg_layers(layer_names):
    """Create a VGG model that returns a list of intermediate output values"""
    # Load pre-trained VGG19
    vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
    vgg.trainable = False

    # Get outputs for specified layers
    outputs = [vgg.get_layer(name).output for name in layer_names]

    # Create model with the specified outputs
    model = tf.keras.Model([vgg.input], outputs)
    return model

def gram_matrix(input_tensor):
    """Calculate Gram matrix for style representation - fixed for graph mode"""
    # Get shape information properly
    shape = tf.shape(input_tensor)
    batch = shape[0]
    height = shape[1]
    width = shape[2]
    channels = shape[3]

    # Reshape the tensor
    features = tf.reshape(input_tensor, (batch, height * width, channels))

    # Calculate Gram matrix
    gram = tf.matmul(features, features, transpose_a=True)

    # Normalize by number of locations
    num_locations = tf.cast(height * width, tf.float32)
    return gram / num_locations

# Content layer where we'll extract feature maps
content_layers = ['block4_conv2']

# Style layers we'll use for style extraction
style_layers = [
    'block1_conv1',
    'block2_conv1',
    'block3_conv1',
    'block4_conv1',
    'block5_conv1'
]

# Number of style layers
num_style_layers = len(style_layers)
num_content_layers = len(content_layers)

# Create feature extraction model based on VGG19
def get_model():
    """Create the feature extraction model"""
    # Combine style and content layers
    all_layers = style_layers + content_layers

    # Build the model
    vgg_model = vgg_layers(all_layers)

    # Return the model
    return vgg_model

print("Feature extraction model defined!")

In [ ]:
class StyleContentModel(tf.keras.models.Model):
    def __init__(self, style_layers, content_layers):
        super(StyleContentModel, self).__init__()
        self.vgg = get_model()
        self.style_layers = style_layers
        self.content_layers = content_layers
        self.num_style_layers = len(style_layers)
        self.vgg.trainable = False

    @tf.function  # Add tf.function decorator to ensure graph mode compatibility
    def call(self, inputs):
        """Extract style and content features"""
        # Scale inputs to VGG19 expected range
        inputs = inputs * 255.0
        preprocessed_input = tf.keras.applications.vgg19.preprocess_input(inputs)

        # Get all outputs from VGG19
        outputs = self.vgg(preprocessed_input)

        # Split into style and content features
        style_outputs, content_outputs = (outputs[:self.num_style_layers],
                                          outputs[self.num_style_layers:])

        # Process style outputs to get Gram matrices
        style_outputs_processed = []
        for style_output in style_outputs:
            style_outputs_processed.append(gram_matrix(style_output))

        # Create dictionaries of outputs
        content_dict = {}
        for i, name in enumerate(self.content_layers):
            content_dict[name] = content_outputs[i]

        style_dict = {}
        for i, name in enumerate(self.style_layers):
            style_dict[name] = style_outputs_processed[i]

        return {'content': content_dict, 'style': style_dict}

# Create an instance of the style-content model
extractor = StyleContentModel(style_layers, content_layers)

# Define global variables for loss weights
style_weight = 1e-2
content_weight = 1e4
total_variation_weight = 30

def style_content_loss(outputs, style_targets, content_targets):
    """Calculate the style and content loss"""
    style_outputs = outputs['style']
    content_outputs = outputs['content']

    # Calculate style loss
    style_loss = tf.add_n([tf.reduce_mean((style_outputs[name] - style_targets[name])**2)
                           for name in style_outputs.keys()])

    # Weight style loss by number of layers
    style_loss *= style_weight / num_style_layers

    # Calculate content loss
    content_loss = tf.add_n([tf.reduce_mean((content_outputs[name] - content_targets[name])**2)
                             for name in content_outputs.keys()])

    # Weight content loss by number of layers
    content_loss *= content_weight / num_content_layers

    return style_loss, content_loss

def total_variation_loss(image):
    """Calculate total variation loss to encourage smoothness"""
    return tf.image.total_variation(image)



def train_step(image, style_targets, content_targets, optimizer):
    """One optimization step without tf.function decorator"""
    with tf.GradientTape() as tape:
        # Extract features
        outputs = extractor(image)

        # Calculate losses
        style_loss, content_loss = style_content_loss(outputs, style_targets, content_targets)

        # Calculate total variation loss for smoothing
        tv_loss = total_variation_loss(image) * total_variation_weight

        # Calculate total loss
        loss = style_loss + content_loss + tv_loss

    # Calculate gradients
    grad = tape.gradient(loss, image)

    # Apply gradients
    optimizer.apply_gradients([(grad, image)])

    # Clip pixel values to [0,1]
    image.assign(clip_0_1(image))

    return loss, style_loss, content_loss, tv_loss

print("Style content extractor and loss functions defined!")

In [ ]:
# Update run_style_transfer to use the non-decorated train_step
def run_style_transfer(content_image, style_image, output_widget,
                      style_weight_exp=1, content_weight_exp=4,
                      tv_weight=30, iterations=500,
                      display_interval=100):
    """Run style transfer with the given parameters and display results in the output widget"""
    global style_weight, content_weight, total_variation_weight

    # Clean output
    output_widget.clear_output()

    with output_widget:
        # Set weights based on parameters
        style_weight = 10.0 ** (style_weight_exp - 2)  # 10^-1 = 0.1 is a good default
        content_weight = 10.0 ** content_weight_exp    # 10^4 = 10000 is a good default
        total_variation_weight = tv_weight

        print(f"Running style transfer with:")
        print(f"- Style weight: {style_weight:.6f} (10^{style_weight_exp-2})")
        print(f"- Content weight: {content_weight:.1f} (10^{content_weight_exp})")
        print(f"- Total variation weight: {total_variation_weight}")
        print(f"- Iterations: {iterations}")

        # Extract style and content features
        style_targets = extractor(style_image)['style']
        content_targets = extractor(content_image)['content']

        # Initialize with content image
        image = tf.Variable(content_image)

        # Create optimizer
        optimizer = tf.optimizers.Adam(learning_rate=0.02, beta_1=0.99, epsilon=1e-1)

        # Store loss values for plotting
        loss_history = []
        style_loss_history = []
        content_loss_history = []

        # Initial display
        print("Starting style transfer...")
        show_images(content_image, style_image, image)

        # Start timer
        start_time = time.time()

        # Optimize
        for i in range(iterations):
            loss, style_loss, content_loss, tv_loss = train_step(
                image, style_targets, content_targets, optimizer)

            # Store losses (ensure they're scalar values)
            loss_history.append(float(loss.numpy()))
            style_loss_history.append(float(style_loss.numpy()))
            content_loss_history.append(float(content_loss.numpy()))

            # Display progress
            if (i+1) % display_interval == 0 or i == 0:
                # Clear previous outputs but keep the history
                output_widget.clear_output(wait=True)

                # Calculate time elapsed
                elapsed = time.time() - start_time
                print(f"Iteration: {i+1}/{iterations}, Time elapsed: {elapsed:.2f}s")
                print(f"Total Loss: {float(loss.numpy()):.4e}, "
                     f"Style Loss: {float(style_loss.numpy()):.4e}, "
                     f"Content Loss: {float(content_loss.numpy()):.4e}, "
                     f"TV Loss: {float(tv_loss.numpy()):.4e}")

                # Display current result
                show_images(content_image, style_image, image)

                # Plot loss history if we have enough data
                if len(loss_history) > 1:
                    plt.figure(figsize=(12, 4))
                    plt.subplot(1, 2, 1)
                    plt.semilogy(loss_history, label='Total Loss')
                    plt.title('Total Loss')
                    plt.xlabel('Iteration')
                    plt.ylabel('Loss (log scale)')

                    plt.subplot(1, 2, 2)
                    plt.semilogy(style_loss_history, label='Style Loss')
                    plt.semilogy(content_loss_history, label='Content Loss')
                    plt.title('Component Losses')
                    plt.xlabel('Iteration')
                    plt.ylabel('Loss (log scale)')
                    plt.legend()

                    plt.tight_layout()
                    plt.show()

        # Final display
        output_widget.clear_output(wait=True)
        print(f"Style transfer completed in {time.time() - start_time:.2f}s")
        show_images(content_image, style_image, image)

        # Save the result
        result_img = tensor_to_image(image)
        output_filename = f"stylized_image_{int(time.time())}.png"
        result_img.save(output_filename)

        # Allow download
        try:
            files.download(output_filename)
            print(f"\nDownload link for '{output_filename}' generated.")
        except:
            print(f"\nImage saved as '{output_filename}'")

        return image

def create_ui_with_button(content_image, style_image):
    """Create interactive UI with a button to trigger style transfer"""
    if content_image is None or style_image is None:
        print("Images not loaded. Please run upload_and_process_images first.")
        return

    # Create output area for results
    output = Output()

    # Create interactive widgets
    style_slider = FloatSlider(
        value=1, min=-1, max=3, step=0.1,
        description='Style (10^x)',
        continuous_update=False,
        style={'description_width': '100px'}
    )

    content_slider = FloatSlider(
        value=4, min=3, max=6, step=0.1,
        description='Content (10^x)',
        continuous_update=False,
        style={'description_width': '100px'}
    )

    tv_slider = IntSlider(
        value=30, min=0, max=100, step=5,
        description='Smoothing',
        continuous_update=False,
        style={'description_width': '100px'}
    )

    iter_slider = IntSlider(
        value=500, min=100, max=2000, step=50,
        description='Iterations',
        continuous_update=False,
        style={'description_width': '100px'}
    )

    display_interval_slider = IntSlider(
        value=100, min=10, max=500, step=5,
        description='Display every',
        continuous_update=False,
        style={'description_width': '100px'}
    )

    # Create start button
    start_button = Button(
        description='Start Style Transfer',
        button_style='success',
        tooltip='Click to start the style transfer process',
        icon='play',
        layout=Layout(width='200px', height='40px')
    )

    # Button click handler
    def on_button_click(b):
        # Disable button during processing
        start_button.disabled = True
        start_button.description = 'Processing...'

        # Run style transfer with current parameters
        try:
            run_style_transfer(
                content_image,
                style_image,
                output,
                style_weight_exp=style_slider.value,
                content_weight_exp=content_slider.value,
                tv_weight=tv_slider.value,
                iterations=iter_slider.value,
                display_interval=display_interval_slider.value
            )
        finally:
            # Re-enable button when done
            start_button.disabled = False
            start_button.description = 'Start Style Transfer'

    # Connect button handler
    start_button.on_click(on_button_click)

    # Display the UI
    print("Adjust parameters and click 'Start Style Transfer' when ready:")

    # Create UI layout
    controls = VBox([
        HBox([style_slider, content_slider]),
        HBox([tv_slider, iter_slider, display_interval_slider]),
        start_button
    ])

    display(controls)
    display(output)

    # Show initial images
    with output:
        print("Preview of input images:")
        show_images(content_image, style_image)

print("Style transfer function with button trigger created!")

In [ ]:
# Reset the model to ensure clean execution
def reset_model():
    """Reset the model and global variables"""
    global extractor, style_weight, content_weight, total_variation_weight

    # Reinitialize the extractor
    extractor = StyleContentModel(style_layers, content_layers)

    # Reset weights to defaults
    style_weight = 1e-2
    content_weight = 1e4
    total_variation_weight = 30

    print("Model and parameters reset to defaults.")

# Recommended parameter presets for quick access
def create_preset_ui(content_image, style_image):
    """UI with preset buttons for common style transfer configurations"""
    if content_image is None or style_image is None:
        print("Images not loaded. Please run upload_and_process_images first.")
        return

    # Create output area for results
    output = Output()

    # Define presets
    presets = {
        "Balanced": {
            'style': 1.0,
            'content': 4.0,
            'tv': 30,
            'iterations': 500,
            'desc': "Standard balance of style and content"
        },
        "Strong Style": {
            'style': 1.5,
            'content': 4.0,
            'tv': 20,
            'iterations': 500,
            'desc': "More pronounced Van Gogh style effect"
        },
        "Clear Face": {
            'style': 0.5,
            'content': 4.5,
            'tv': 20,
            'iterations': 500,
            'desc': "Better face preservation with subtle style"
        },
        "Artistic": {
            'style': 2.0,
            'content': 3.5,
            'tv': 40,
            'iterations': 800,
            'desc': "Creative interpretation, more like a painting"
        }
    }

    # Create buttons for each preset
    preset_buttons = {}
    for name, params in presets.items():
        btn = Button(
            description=name,
            tooltip=params['desc'],
            button_style='info',
            layout=Layout(width='150px', height='40px')
        )
        preset_buttons[name] = btn

    # Button click handler for presets
    def on_preset_click(b):
        # Get the preset name from button description
        preset_name = b.description
        preset = presets[preset_name]

        # Disable all buttons during processing
        for btn in preset_buttons.values():
            btn.disabled = True
            if btn.description == preset_name:
                btn.description = "Processing..."

        # Run style transfer with preset parameters
        try:
            run_style_transfer(
                content_image,
                style_image,
                output,
                style_weight_exp=preset['style'],
                content_weight_exp=preset['content'],
                tv_weight=preset['tv'],
                iterations=preset['iterations'],
                display_interval=100
            )
        finally:
            # Re-enable buttons when done
            for name, btn in preset_buttons.items():
                btn.disabled = False
                btn.description = name

    # Connect button handlers
    for btn in preset_buttons.values():
        btn.on_click(on_preset_click)

    # Create custom UI button
    custom_btn = Button(
        description='Custom Settings',
        tooltip='Use custom parameter settings',
        button_style='success',
        layout=Layout(width='150px', height='40px')
    )

    def on_custom_click(b):
        create_ui_with_button(content_image, style_image)

    custom_btn.on_click(on_custom_click)

    # Display the UI
    print("Select a preset style transfer configuration or use custom settings:")

    # Create preset buttons layout
    preset_layout = HBox(list(preset_buttons.values()))

    # Display UI components
    display(VBox([
        HBox([preset_layout]),
        HBox([custom_btn])
    ]))
    display(output)

    # Show initial images
    with output:
        print("Preview of input images:")
        show_images(content_image, style_image)

# Main execution point
print("\nNeural Style Transfer System Ready!")
print("\nExecute the following command to start:")
print("content_image, style_image = upload_and_process_images()")
print("\nThen choose between:")
print("create_ui_with_button(content_image, style_image)  # For full custom UI")
print("create_preset_ui(content_image, style_image)       # For preset configurations")

In [ ]:
content_image, style_image = upload_and_process_images()

In [ ]:
create_ui_with_button(content_image, style_image)